<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/lotus_online_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title install command (Optimized)
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 80px;
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 13px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting Lotus's environment setup...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        # Inform the user which command is currently running
        display(HTML(f"""<script>
            var obj = document.getElementById('scroll_output');
            obj.innerHTML += '<div style="color: #ffaa00;">Running: {cmd.split()[0]}...</div>';
            obj.scrollTop = obj.scrollHeight;
        </script>"""), display_id='scroll_update')

        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            escaped_line = line.replace("'", "\\'").strip()
            # Only print lines that actually contain text (ignores blank lines from quiet outputs)
            if escaped_line:
                display(HTML(f"""<script>
                    var obj = document.getElementById('scroll_output');
                    obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                    obj.scrollTop = obj.scrollHeight;
                </script>"""), display_id='scroll_update')
        process.wait()

# 2. Optimized & Consolidated Commands
commands_to_run = [
    # System Updates and Chrome Installation (-qq makes it extra quiet to save UI lag)
    "apt-get -y -qq update",
    "apt-get install -y -qq google-chrome-stable",

    # Python Library Installation (Combined into one call to resolve dependencies once)
    "pip install selenium bs4 pandas chromedriver-autoinstaller polars xlsxwriter fastexcel playwright beautifulsoup4 itables curl_cffi msgspec browserforge nest_asyncio scrapling patchright -q",

    # Browser Binaries Installation (Dropped standard Playwright, kept only Patchright)
    "patchright install chromium",
    "patchright install-deps",
    "playwright install chromium",  # <-- Add this
    "playwright install-deps"       # <-- Add this
]

run_and_scroll(commands_to_run)
print("\n✅ Environment ready! Let's go.")


✅ Environment ready! Let's go.


### Import Lib

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-05-20


In [3]:
# @title udf scrape lotus (Catalog Condition Added + Retry Queue)
import asyncio
import polars as pl
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from collections import deque

async def scrape_lotuss_scroller(shop_url_list: list):
    all_extracted_data = []

    # Initialize a queue: storing tuples of (url, current_retry_count)
    url_queue = deque([(url, 0) for url in shop_url_list])
    MAX_RETRIES = 3  # Safety net to prevent infinite loops on broken URLs

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        # Process the queue until it's empty
        while url_queue:
            # Dequeue the next URL and its attempt count
            shop_url, attempts = url_queue.popleft()

            print(f"\n🚀 Opening catalog: {shop_url} (Attempt {attempts + 1})")
            try:
                await page.goto(shop_url, wait_until="domcontentloaded", timeout=60000)
                # Wait for the first batch of products to load
                await page.wait_for_selector(".mui-style-17swnep", timeout=15000)

            except Exception as e:
                print(f"⚠️ Failed to load products for this URL.")
                if attempts < MAX_RETRIES:
                    print("🔄 Appending URL back to the queue to try again later...")
                    url_queue.append((shop_url, attempts + 1))
                    await asyncio.sleep(2) # Brief pause before moving to the next item in queue
                else:
                    print(f"❌ Max retries ({MAX_RETRIES}) reached. Skipping this URL permanently.")

                # Move to the next item in the while loop
                continue

            # --- THE INFINITE SCROLLER ---
            previous_item_count = 0
            scroll_attempts = 0
            max_scrolls = 30 # Safety net

            while scroll_attempts < max_scrolls:
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await asyncio.sleep(4)

                current_items = await page.query_selector_all(".MuiCard-root")
                current_count = len(current_items)

                print(f"  ↳ Scroll {scroll_attempts + 1}: Found {current_count} products on screen...")

                if current_count == previous_item_count:
                    print("  ↳ Hit the bottom of the catalog! Extracting data...")
                    break

                previous_item_count = current_count
                scroll_attempts += 1

            # Extract the fully loaded HTML for this specific URL
            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")
            products = soup.find_all("div", class_="MuiCard-root")

            for prod in products:
                # 1. Extract Name
                name_elem = prod.find(class_="mui-style-17swnep")
                name = name_elem.text.strip() if name_elem else None

                if not name:
                    continue

                # 2. Extract Promotion Price
                promo_elem = prod.find("div", class_="mui-style-18s6ztp")
                if promo_elem:
                    raw_promo = promo_elem.text.replace(',', '').replace('฿', '').strip()
                    promotion_price = raw_promo if raw_promo else None
                else:
                    promotion_price = None

                # 3. Extract Original Price
                orig_elem = prod.find("div", class_="mui-style-f59hd5")
                if orig_elem:
                    raw_orig = orig_elem.text.replace(',', '').replace('฿', '').strip()
                    original_price = raw_orig if raw_orig else promotion_price
                else:
                    original_price = promotion_price

                # 4. Extract Condition
                condition_elem = prod.find("div", class_="mui-style-axryyp")
                if condition_elem:
                    # Extracts text from all the nested <span> tags, joining them cleanly
                    condition = " ".join(condition_elem.get_text(" ").split())
                else:
                    condition = None

                all_extracted_data.append({
                    "name": name,
                    "promotion_price": promotion_price,
                    "original_price": original_price,
                    "condition": condition
                })

            # Short sleep before hitting the next URL to be nice to the server
            await asyncio.sleep(3)

        await browser.close()

    # Create the Master DataFrame from all URLs
    df = pl.DataFrame(all_extracted_data)

    # Convert Data Types and cleanup duplicates across all categories
    if not df.is_empty():
        df = df.with_columns([
            pl.col("name").cast(pl.String),
            pl.col("promotion_price").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.Float64, strict=False),
            pl.col("condition").cast(pl.String)
        ])
        # Deduplicate in case the same item appears in multiple categories
        df = df.unique(subset=["name"], maintain_order=True)

    return df

In [8]:
# Define your categories
catalog_urls = [
    "https://www.lotuss.com/en/category/household-and-merits/86590-laundry-supplies?sort=relevance:DESC&filter.brandId=21490,21430,21829,22054,23485",

    "https://www.lotuss.com/en/category/household-and-merits/86590-cleaning-chemical?sort=relevance:DESC&filter.categoryId=87100&filter.brandId=22327,24049"
]

# Run the master scraper
df_lotuss_catalog = await scrape_lotuss_scroller(shop_url_list=catalog_urls)

# Check the results
print(f"\n--- Master Catalog Scrape Complete! Total unique items: {len(df_lotuss_catalog)} ---")
print(df_lotuss_catalog)


🚀 Opening catalog: https://www.lotuss.com/en/category/household-and-merits/86590-laundry-supplies?sort=relevance:DESC&filter.brandId=21490,21430,21829,22054,23485 (Attempt 1)
  ↳ Scroll 1: Found 60 products on screen...
  ↳ Scroll 2: Found 90 products on screen...
  ↳ Scroll 3: Found 120 products on screen...
  ↳ Scroll 4: Found 150 products on screen...
  ↳ Scroll 5: Found 180 products on screen...
  ↳ Scroll 6: Found 210 products on screen...
  ↳ Scroll 7: Found 240 products on screen...
  ↳ Scroll 8: Found 270 products on screen...
  ↳ Scroll 9: Found 300 products on screen...
  ↳ Scroll 10: Found 330 products on screen...
  ↳ Scroll 11: Found 360 products on screen...
  ↳ Scroll 12: Found 390 products on screen...
  ↳ Scroll 13: Found 420 products on screen...
  ↳ Scroll 14: Found 450 products on screen...
  ↳ Scroll 15: Found 480 products on screen...
  ↳ Scroll 16: Found 510 products on screen...
  ↳ Scroll 17: Found 540 products on screen...
  ↳ Scroll 18: Found 570 products on

In [9]:
df_lotuss_catalog.to_pandas()

,name,promotion_price,original_price,condition
0,PAO SILVER NANO ACTIVE DETERGENT 750G.,69.0,69.0,None
1,PAO SILVER NANO DETERGENT800G,69.0,69.0,None
2,PAO SILVER NANO 800G.MACHINE FRONT LOAD,69.0,69.0,None
3,PAO SILVER NANO SOFT DETERGENT 800 G.,69.0,69.0,None
4,FINELINE LIQUID DETERGENT COOLING FRESH 1200 ML,179.0,179.0,None
...,...,...,...,...
592,LIPON F DISHWASH HYGIENIC REFILL 750 ML PACK 2,69.0,85.0,None
593,LIPON F PURIFY ROSEMARY 460ML,89.0,89.0,None
594,LIPON F DISHWASHING KAFFIR LIME 750ML. PACK 2,69.0,82.0,None
595,LIPON F PURIFY ALOEVERA PEPERMINT 460ML,89.0,89.0,None


In [10]:
# @title
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_lotuss = re_evaluate_price(df_lotuss_catalog)

In [11]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    # Updated pattern:
    # 1. Added (?)i for case-insensitivity
    # 2. Removed \b to ensure Thai characters don't get blocked by boundary logic
    quant_unit_pattern = r"(?i)([\d.]+)\s*(ML|G|KG|L|กรัม|GRAMS?)"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(date.today()).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [12]:
# 2. Transform Lotus's (from yesterday's script)
df_trans_lotuss = parse_product_names(df_prep_lotuss, shop_name = "Lotus's")

In [13]:
# @title write Excel
df_trans_lotuss.write_excel(f"lotus_{today_date}.xlsx")

### Search watchlist

In [14]:
list_to_search = [
# --- Lotus's ---
'FINELINE LIQUID PLUS GOLD 550 ML.',
'FINELINE LIQUID PLUS GOLD 1250 ML.',
'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.',
'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.',
'PAO WIN WASH LIQUID REFILL 620 ML',
'PAO WIN WASH LIQUID 1300 ML',
'PAO SUPER WHITE POWDER DETERGENT1800G.P2',
'PAO SUPER WHITE POWDER DETERGENT 2400 G',
'ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK',
'ATTACK EASY HAPPY SWEET DETERGENT 2500G.',
'PRO POWER DETERGENT BLUE PLUS 1700 G. PACK 1+1',
'PRO POWDER DETERGENT BLUE PLUS 2400 G.',
'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUCH 480 ML.',
'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHITE 480 ML 2+1',
'HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 1000 ML',
'HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH 1000 ML 1+1',
'LIPON F DISHWASHING HYGIENE 500 ML PACK 3',
'LIPON F DISH WASHER XTRA HYGENIC 750 ML. PACK 2',
'LIPON-F DISHWASH BERGAMOT GALLON 3200 ML',
]
search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_lotuss to get original_price and promotion_price
search_results_df = search_df.join(
    df_trans_lotuss.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
lotuss_names_set = set(df_trans_lotuss["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(lotuss_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)
search_results_df.to_pandas()

Search Results with Prices:
shape: (19, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ f64            ┆ f64             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ FINELINE LIQUID PLUS GOLD 550 … ┆ null           ┆ null            ┆ false │
│ PAO SUPER WHITE POWDER DETERGE… ┆ 155.0          ┆ null            ┆ true  │
│ PAO SUPER WHITE POWDER DETERGE… ┆ null           ┆ null            ┆ false │
│ HYGIENE FABRIC SOFTENER EXPERT… ┆ 60.0           ┆ 49.0            ┆ true  │
│ PAO WIN WASH LIQUID REFILL 620… ┆ null           ┆ null            ┆ false │
│ …                               ┆ …              ┆ …               ┆ …     │
│ HYGIENE FABRIC SOFTENER EXPERT… ┆ 219.0          ┆ 195.0           ┆ true  │
│ LIPON-F

,product_name,original_price,promotion_price,Found
0,FINELINE LIQUID PLUS GOLD 550 ML.,NaN,NaN,False
1,PAO SUPER WHITE POWDER DETERGENT 2400 G,155.0,NaN,True
2,PAO SUPER WHITE POWDER DETERGENT1800G.P2,NaN,NaN,False
3,HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUC...,60.0,49.0,True
4,PAO WIN WASH LIQUID REFILL 620 ML,NaN,NaN,False
5,HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.,139.0,114.0,True
6,HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHIT...,NaN,NaN,False
7,PAO WIN WASH LIQUID 1300 ML,185.0,NaN,True
8,FINELINE LIQUID PLUS GOLD 1250 ML.,NaN,NaN,False
9,HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 100...,129.0,109.0,True


In [15]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return # Does nothing, skipping the cell content

In [16]:
# @title watch not code
%%skip
[
# --- Lotus's ---
# 'FINELINE LIQUID PLUS GOLD 550 ML.'
"https://www.lotuss.com/en/product/75552978",
# 'FINELINE LIQUID PLUS GOLD 1250 ML.'
"https://www.lotuss.com/en/product/52564595",
# 'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.',
"https://www.lotuss.com/en/product/75605072",
# 'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.',
"https://www.lotuss.com/en/product/75605074",
# 'PAO WIN WASH LIQUID REFILL 620 ML',
"https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-700ml-27074455",
# 'PAO WIN WASH LIQUID 1300 ML',
"https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-1500ml-74515209",
# 'PAO SUPER WHITE POWDER DETERGENT1800G.P2',
"https://www.lotuss.com/en/product/51625744"
# 'PAO SUPER WHITE POWDER DETERGENT 2400 G',
"https://www.lotuss.com/en/product/pao-super-white-standard-formula-for-hand-wash-top-loading-washing-machine-2700g-176389",
# 'ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK',
"https://www.lotuss.com/en/product/75659870",
# 'ATTACK EASY HAPPY SWEET DETERGENT 2500G.',
"https://www.lotuss.com/en/product/attack-easy-happy-sweet-conventional-detergent-2700g-75278235",
# 'PRO POWER DETERGENT BLUE PLUS 1700 G. PACK 1+1',
"https://www.lotuss.com/en/product/75551723",
# 'PRO POWDER DETERGENT BLUE PLUS 2400 G.',
"https://www.lotuss.com/en/product/pro-powder-detergent-2700-g-5314917",
# 'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUCH 480 ML.',
"https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-540ml-71833501",
# 'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHITE 480 ML 2+1',
"https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-540ml-x-21pcs-72884576",
# 'HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 1000 ML',
"https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-1300ml-50625999",
# 'HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH 1000 ML 1+1',
"https://www.lotuss.com/en/product/hygiene-e-pert-milky-white-1300ml-p1-1-51586608",
# 'LIPON F DISHWASHING HYGIENE 500 ML PACK 3',
"https://www.lotuss.com/en/product/lipon-f-refill-concentrated-dishwashing-liquid-550ml-x-3pcs-12942545",
# 'LIPON F DISH WASHER XTRA HYGENIC 750 ML. PACK 2',
"https://www.lotuss.com/en/product/75645272",
# 'LIPON-F DISHWASH BERGAMOT GALLON 3200 ML',
"https://www.lotuss.com/en/product/75645274"
]

In [17]:
# @title udf Scrape Watchlist (Sequential with Retry Queue)
import asyncio
import polars as pl
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def scrape_lotuss_watchlist_sequential(urls: list):
    final_data = []
    retry_queue = []

    async with async_playwright() as p:
        # Anti-bot arguments
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        # Using a single context sequentially helps maintain session cookies!
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )

        # We reuse the exact same tab for every request, just like a real human browsing
        page = await context.new_page()

        async def process_url(url):
            print(f"🚀 Processing: {url}")
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                await asyncio.sleep(4) # Give JS time to render prices

                html_content = await page.content()
                soup = BeautifulSoup(html_content, "html.parser")

                # Extract Name
                name_elem = soup.find("h1", class_="mui-style-1wc9qn4") or soup.find("h1")
                name = name_elem.text.strip() if name_elem else "Unknown Product"

                # Detect Blockers
                if "Privacy Preference Center" in name or name == "Unknown Product":
                    return None, False

                # Extract Prices
                current_price_elem = soup.find("div", class_="mui-style-kbur42")
                crossed_price_elem = soup.find("div", class_="mui-style-1opqjiq")

                raw_current = current_price_elem.get_text().replace(',', '').replace('฿', '').split('/')[0].strip() if current_price_elem else None
                raw_crossed = crossed_price_elem.text.replace(',', '').replace('฿', '').strip() if crossed_price_elem else None

                if raw_crossed:
                    promo_price, orig_price = raw_current, raw_crossed
                else:
                    promo_price, orig_price = None, raw_current

                # Extract Condition
                condition_elem = soup.find("div", class_="mui-style-abwb4l")
                condition = condition_elem.get_text(separator=" ").strip() if condition_elem else None

                return {
                    "url": url,
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": orig_price,
                    "condition": condition
                }, True

            except Exception as e:
                print(f"❌ Error on {url}: {e}")
                return None, False

        # --- 1st Pass: Main List ---
        print("--- Starting First Pass ---")
        for url in urls:
            data, success = await process_url(url)

            if success:
                final_data.append(data)
            else:
                print(f"⚠️ Blocked or failed. Adding to retry queue: {url}")
                retry_queue.append(url)

            # Short delay between sequential requests to avoid triggering the firewall
            await asyncio.sleep(2)

        # --- 2nd Pass: Retry Queue ---
        if retry_queue:
            print(f"\n--- Starting Retry Queue ({len(retry_queue)} items) ---")
            await asyncio.sleep(8) # Let the server cool down for a bit before retrying

            for url in retry_queue:
                data, success = await process_url(url)

                if success:
                    final_data.append(data)
                else:
                    print(f"❌ Permanently failed on retry: {url}")
                    # Keep a record of the failure so the row isn't completely lost
                    final_data.append({
                        "url": url,
                        "name": "Blocked by Cookie Banner",
                        "promotion_price": None,
                        "original_price": None,
                        "condition": None
                    })

                await asyncio.sleep(3)

        await browser.close()

      # Create the initial DataFrame from the scraped list of dictionaries
        df = pl.DataFrame(final_data)

        # Explicitly enforce the requested Data Types
        df = df.with_columns([
            pl.col("url").cast(pl.String),
            pl.col("name").cast(pl.String),
            # strict=False ensures that if a weird string slips through, it safely becomes Null instead of crashing
            pl.col("promotion_price").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.Float64, strict=False),
            pl.col("condition").cast(pl.String)
        ])

        return df

In [19]:
# @title RUN Watchlist Scraper
watchlist_urls = [
    "https://www.lotuss.com/en/product/75552978",
    "https://www.lotuss.com/en/product/52564595",
    "https://www.lotuss.com/en/product/75605072",
    "https://www.lotuss.com/en/product/75605074",
    "https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-700ml-27074455",
    "https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-1500ml-74515209",
    "https://www.lotuss.com/en/product/51625738",
    "https://www.lotuss.com/en/product/51625744",
    "https://www.lotuss.com/en/product/pao-super-white-standard-formula-for-hand-wash-top-loading-washing-machine-2700g-176389",
    "https://www.lotuss.com/en/product/75659870",
    "https://www.lotuss.com/en/product/attack-easy-happy-sweet-conventional-detergent-2700g-75278235",
    "https://www.lotuss.com/en/product/75551723",
    "https://www.lotuss.com/en/product/pro-powder-detergent-2700-g-5314917",
    "https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-540ml-71833501",
    "https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-540ml-x-21pcs-72884576",
    "https://www.lotuss.com/en/product/hygiene-expert-care-milky-touch-refill-concentrate-fabric-softener-1300ml-50625999",
    "https://www.lotuss.com/en/product/hygiene-e-pert-milky-white-1300ml-p1-1-51586608",
    "https://www.lotuss.com/en/product/lipon-f-refill-concentrated-dishwashing-liquid-550ml-x-3pcs-12942545",
    "https://www.lotuss.com/en/product/75645272",
    "https://www.lotuss.com/en/product/75645274",
    "https://www.lotuss.com/en/product/lipon-f-xtra-clean-kaffir-lime-formula-dishwashing-liquid-3600ml-73289973",
]

# Run the sequential queue version
df_watchlist_final = await scrape_lotuss_watchlist_sequential(watchlist_urls)

print("\n--- Watchlist Scrape Complete ---")
print(df_watchlist_final)

--- Starting First Pass ---
🚀 Processing: https://www.lotuss.com/en/product/75552978
🚀 Processing: https://www.lotuss.com/en/product/52564595
🚀 Processing: https://www.lotuss.com/en/product/75605072
🚀 Processing: https://www.lotuss.com/en/product/75605074
🚀 Processing: https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-700ml-27074455
🚀 Processing: https://www.lotuss.com/en/product/pao-win-wash-liquid-refill-concentrated-liquid-detergent-1500ml-74515209
🚀 Processing: https://www.lotuss.com/en/product/51625738
🚀 Processing: https://www.lotuss.com/en/product/51625744
🚀 Processing: https://www.lotuss.com/en/product/pao-super-white-standard-formula-for-hand-wash-top-loading-washing-machine-2700g-176389
🚀 Processing: https://www.lotuss.com/en/product/75659870
🚀 Processing: https://www.lotuss.com/en/product/attack-easy-happy-sweet-conventional-detergent-2700g-75278235
🚀 Processing: https://www.lotuss.com/en/product/75551723
🚀 Processing: https://www.lot

In [20]:
df_watchlist_final.to_pandas()

,url,name,promotion_price,original_price,condition
0,https://www.lotuss.com/en/product/75552978,FINELINE LIQUID PLUS GOLD 550 ML.,NaN,89.0,Buy 1 get 1 free Please add only buying quanti...
1,https://www.lotuss.com/en/product/52564595,FINELINE LIQUID PLUS GOLD 1250 ML.,NaN,179.0,Buy 1 get 1 free Please add only buying quanti...
2,https://www.lotuss.com/en/product/75605072,HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.,49.0,65.0,None
3,https://www.lotuss.com/en/product/75605074,HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.,114.0,139.0,Buy 1 Unit Get 3 Coins
4,https://www.lotuss.com/en/product/pao-win-wash...,PAO WIN WASH LIQUID REFILL 620 ML,NaN,99.0,None
5,https://www.lotuss.com/en/product/pao-win-wash...,PAO WIN WASH LIQUID 1300 ML,NaN,185.0,Buy 1 get 1 free Please add only buying quanti...
6,https://www.lotuss.com/en/product/51625738,PAO SUPER WHITE POWDER DETERGENT1800G.P2,155.0,169.0,None
7,https://www.lotuss.com/en/product/51625744,PAO SUPER SOFTPOWDER DETERGENT1800G.P2,155.0,169.0,None
8,https://www.lotuss.com/en/product/pao-super-wh...,PAO SUPER WHITE POWDER DETERGENT 2400 G,NaN,155.0,None
9,https://www.lotuss.com/en/product/75659870,ATTACK EASY POWDER HAPPY SWEET 1700 G TWIN PACK,149.0,169.0,None


In [21]:
list_to_search = [
# --- Lotus's ---
'FINELINE LIQUID PLUS GOLD 550 ML.',
'FINELINE LIQUID PLUS GOLD 1250 ML.',
'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.',
'HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.',
'PAO WIN WASH LIQUID REFILL 620 ML',
'PAO WIN WASH LIQUID 1300 ML',
'PAO SUPER WHITE POWDER DETERGENT1800G.P2',
'PAO SUPER WHITE POWDER DETERGENT 2400 G',
'ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK',
'ATTACK EASY HAPPY SWEET DETERGENT 2500G.',
'PRO POWER DETERGENT BLUE PLUS 1700 G. PACK 1+1',
'PRO POWDER DETERGENT BLUE PLUS 2400 G.',
'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUCH 480 ML.',
'HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHITE 480 ML 2+1',
'HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 1000 ML',
'HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH 1000 ML 1+1',
'LIPON F DISHWASHING HYGIENE 500 ML PACK 3',
'LIPON F DISH WASHER XTRA HYGENIC 750 ML. PACK 2',
'LIPON-F DISHWASH BERGAMOT GALLON 3200 ML',
]
search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_lotuss to get original_price and promotion_price
search_results_df = search_df.join(
    df_watchlist_final.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
lotuss_names_set = set(df_watchlist_final["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(lotuss_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)
search_results_df.to_pandas()

Search Results with Prices:
shape: (19, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ f64            ┆ f64             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ ATTACK EASY POWDER HAPPY SWEET… ┆ null           ┆ null            ┆ false │
│ HYGIENE LAUNDRY EXPERT WASH MI… ┆ 139.0          ┆ 114.0           ┆ true  │
│ ATTACK EASY HAPPY SWEET DETERG… ┆ 155.0          ┆ 99.0            ┆ true  │
│ PAO SUPER WHITE POWDER DETERGE… ┆ 169.0          ┆ 155.0           ┆ true  │
│ HYGIENE FABRIC SOFTENER EXPERT… ┆ 219.0          ┆ 195.0           ┆ true  │
│ …                               ┆ …              ┆ …               ┆ …     │
│ PRO POWDER DETERGENT BLUE PLUS… ┆ 150.0          ┆ 99.0            ┆ true  │
│ FINELIN

,product_name,original_price,promotion_price,Found
0,ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK,NaN,NaN,False
1,HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.,139.0,114.0,True
2,ATTACK EASY HAPPY SWEET DETERGENT 2500G.,155.0,99.0,True
3,PAO SUPER WHITE POWDER DETERGENT1800G.P2,169.0,155.0,True
4,HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH...,219.0,195.0,True
5,LIPON-F DISHWASH BERGAMOT GALLON 3200 ML,165.0,NaN,True
6,LIPON F DISHWASHING HYGIENE 500 ML PACK 3,84.0,82.0,True
7,HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.,65.0,49.0,True
8,FINELINE LIQUID PLUS GOLD 550 ML.,89.0,NaN,True
9,HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHIT...,120.0,119.0,True


In [22]:
search_results_df.write_excel(f"lotus_search_result_{today_date}.xlsx")

In [23]:
df_prep_watchlist = re_evaluate_price(df_watchlist_final)
df_trans_watchlist = parse_product_names(df_prep_watchlist, shop_name = "Lotus's")
df_trans_watchlist.write_excel(f"lotus_watchlist_{today_date}.xlsx")